<img style="float: right;" src="img/python.png">

# Teil 8: Datenbanken


## Vorbereitung:
Wir tasten uns an das Thema Datenbanken heran, indem wir uns SQLite bedienen. Diese Datenbank funktioniert ähnlich wie eine SQL-Datenbank wie MySQL oder Mariadb.<br>
Zur Übung können wir in diesen Fall zum warm werden erstmal eine Datenbank mithilfe von PyCharm erstellen. Gehen Sie bitte folgendermaßen vor. Fügen Sie eine Datenbank hinzu:<br>
<img height="536" src="img/sqlite3.png" width="390"/><br>
Erstellen Sie bitte eine neue Datenbank, diese nennen Sie bitte "**telefonbuch.sqlite**".<br><br>
<img style="float: right;" src="img/sqlite4.png">
<br>
Erstellen Sie jetzt bitte eine neue Tabelle "Liste" mit den folgenden Spalten:
<br> 
<img style="float: right;" src="img/sqlite5.png">
Die Datenbank soll die folgende Struktur aufweisen: <br>
<img style="float: right;" src="img/sqlite6.png"><br>
In diesem Fenster wird im unteren Bereich der SQL-Befehl angezeigt, der ausgeführt wird, 
damit die Tabellen angelegt werden:

Diesen kann man an dieser Stelle gut kopieren, um ihn später im Skript zum Erstellen einer neuen Datenbank zu nutzen, sollte keine vorhanden sein.
<br>
<img style="float: right;" src="img/sqlite6.png"><br>
<br>

In [ ]:
%%sql
create table telefonbuch
(
    id        integer
        constraint id
            primary key,
    vorname   TEXT,
    nachname  TEXT,
    vorwahl   TEXT,
    rufnummer TEXT
);

Dieser Code kann beispielsweise dafür genutzt werden, um beim Start eines Programms bei nichtvorhandensein der Datenbank eine neue Datenbank zu erstellen. <br>
zum Beispiel:

In [ ]:
import sqlite3

connection = sqlite3.connect("telefonbuch_connection.db") # Genera la conexion
cursor = connection.cursor() # se genera un cursor con acceso al db

# la orden/Befehl-SQL SE GUARDA en una variable ("sql")
sql = """create table telefonbuch
          (
              id        integer
                  constraint id
                      primary key,
              vorname   TEXT,
              nachname  TEXT,
              vorwahl   TEXT,
              rufnummer TEXT
          )
      """

cursor.execute(sql) # "sql" run
connection.commit() # se confirma la orden/Befehl de nuevo (necesario)
connection.close() # se cierra la conexion


Wenn Sie die Datenbank vorher löschen und dann wieder mit dem letzten Skript neu erstellen, können Sie den Erfolg rechts im Datenbank Browser von PyCharm überprüfen.

#### Datenbankpfad relativ zum Skriptpfad erstellen, empfohlen!

In [ ]:
from pathlib import Path
import sqlite3

# Pfad-Variable abhängig vom Skriptordner erstellen:
ordner = Path(__file__).parent
db_name = ordner / "telefonbuch.db"

conn = sqlite3.connect(db_name)  # Verbindung wird über Pfadvariable hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

# Der gespeicherte SQL-Befehl, von gerade, wird in eine Variable gespeichert

sql = """create table if not exists telefonbuch
(
    id        integer
        constraint id
            primary key,
    vorname   TEXT,
    nachname  TEXT,
    vorwahl   TEXT,
    rufnummer TEXT
)
"""

c.execute(sql)  # Der SQL-Befehl wird ausgeführt
conn.commit()  # Befehl nochmal bestätigt (ist bei allen Schreib-Vorgängen erforderlich)
conn.close()  # Verbindung wird geschlossen

## Erste kleine Datenbankoperationen

### INSERT
Einfügen eines Namens und einer Rufnummer in unsere niegelnagelneuen Datenbank:

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

# Der gespeicherte SQL-Befehl, von gerade, wird in eine Variable gespeichert

var_vorname = "Hans"
var_nachname = "Müller"
var_vorwahl = "0151"
var_rufnummer = "1234567"

params = (var_vorname, var_nachname, var_vorwahl, var_rufnummer)  # Reihenfolge beachten! GANZ WICHTIG!
# params = (var_vorname, )  # Übergabe IMMER als TUPEL! Auch bei einem einzigen Parameter

sql = """INSERT INTO telefonbuch (vorname, nachname, vorwahl, rufnummer) VALUES (?, ?, ?, ?)"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
conn.commit()  # Befehl nochmal bestätigt (ist bei allen Schreib-Vorgängen erforderlich)
conn.close()  # Verbindung wird geschlossen

**Wichtig! Die Parameter müssen immer als Tupel übergeben werden!**

Bei ***nur einem Parameter*** würde das so aussehen:
<code>params = (var_vorname, ) </code>

Vielleicht ist Ihnen aufgefallen, das wir die Variablen nicht direkt in die Abfrage schreiben, sondern als Parameter übergeben. Das hat einen Sicherheitsaspekt, dass zum Beispiel bei Webprogrammierung nicht einfach irgendwelche Werte als SQL-Injektion übergeben werden können.

<br>
Erfolgskontrolle:<br>
<img style="float: right;" src="img/sqlite7.png">
<br>
...bingo!

### SELECT
Damit die folgende Übung mehr Spaß macht, sollten noch ein paar Namen hinzugefügt werden (**Tipp**: Das geht ganz schnell, wenn man sich die Tabelle mit den Datenbankbrowser unter PyCharm auf macht):
<img style="float: right;" src="img/sqlite8.png">


#### Weitere Daten hinzufügen

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

# Der gespeicherte SQL-Befehl, von gerade, wird in eine Variable gespeichert

params = [
("Otto", "Nashorn", "040", "12345"),
("Hans", "Meiser", "030", "65465"),
("Rita", "Müller", "017", "98765"),
]

sql = """INSERT INTO telefonbuch (vorname, nachname, vorwahl, rufnummer) VALUES (?, ?, ?, ?)"""

c.executemany(sql, params)  # Der SQL-Befehl wird ausgeführt
conn.commit()  # Befehl nochmal bestätigt (ist bei allen Schreib-Vorgängen erforderlich)
conn.close()  # Verbindung wird geschlossen

#### Alle Inhalte ausgeben:
Die Ausgabe ist auf den ersten Blick ein wenig "verwirrend", auf den zweiten Blick erkennen wir (hoffentlich), das es sich um eine Liste von Tupel handelt, also iterieren wir das Ganze mal durch:

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

sql = """SELECT * FROM telefonbuch"""  # * = Jede Spalte

c.execute(sql)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()  # Verbindung wird geschlossen

print(ausgabe)  # Ausgabe als Liste mit Tupeln

# Iteration1:
for i in ausgabe:
    print(i)

print()  # Leerzeile

# Iteration2:
for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

print()  # Leerzeile

# Iteration3 (ohne ID):
for _, v, n, r, t in ausgabe:
    print(v, n, r, t)

#### Suche alle Einträge mit dem Nachnamen Müller

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

params = ("Müller", )

sql = """SELECT * FROM telefonbuch WHERE nachname = ?"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()  # Verbindung wird geschlossen

for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

Suche alle Müllers ohne ID:

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

params = ("Müller", )

sql = """SELECT vorname, nachname, vorwahl, rufnummer FROM telefonbuch WHERE nachname = ?"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()  # Verbindung wird geschlossen

for v, n, r, t in ausgabe:
    print(v, n, r, t)

#### Suche nach id mit fetchone()

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

params = (2, )

sql = """SELECT * FROM telefonbuch WHERE id = ?"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchone()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()  # Verbindung wird geschlossen

print(ausgabe)

Tupel direkt entpacken:

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

params = (2, )

sql = """SELECT * FROM telefonbuch WHERE id = ?"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
id, vorname, nachname, vorwahl, rufnummer = c.fetchone()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()  # Verbindung wird geschlossen

print(id)
print(vorname)
print(nachname)
print(vorwahl)
print(rufnummer)

#### Wildcards
Wildcards helfen bei der Suche als Platzhalter.

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

suchbegriff = "Mü%"  # ein %-Zeichen steht für eine beliebige Zeichenfolge oder kein Zeichen

sql = """SELECT * FROM telefonbuch WHERE nachname LIKE ?"""  # Like-Operator: Suche nach einem Muster, ignoriere Gross- und Kleinschreibung

params = (suchbegriff,)
c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()
for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)


In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)


suchbegriff = "%"+ input("Bitte Suchbegriff eingeben: ") + "%"

sql = """SELECT * FROM telefonbuch WHERE nachname LIKE ?"""  # Like-Operator: Suche nach einem Muster, ignoriere Gross- und Kleinschreibung

params = (suchbegriff,)
c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()
for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)


suchbegriff = "%"+ input("Bitte Suchbegriff eingeben: ") + "%"

sql = """SELECT * FROM telefonbuch WHERE nachname LIKE ? OR vorname LIKE ? """  # Like-Operator: Suche nach einem Muster, ignoriere Gross- und Kleinschreibung

params = (suchbegriff, suchbegriff)
c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()

for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

#### Wildcard für 1 Zeichen

In [1]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

suchbegriff = "M_ller"
params = (suchbegriff,)
sql = """SELECT * FROM telefonbuch WHERE nachname LIKE ?"""  # Like-Operator: Suche nach einem Muster, ignoriere Gross- und Kleinschreibung

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.close()


for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

1 Christian Müller 0176 1234567


## DELETE Daten löschen

Daten löschen gehört in Datenbanken in der Regel dazu... <br> <br>
Als erstes lassen wir uns nochmal den Inhalt der Tabelle "**Liste**" anzeigen. <br>
Zur besseren Übersicht mal den **vollständigen** Code um den kompletten Inhalt der Datenbank anzuzeigen:

Jetzt löschen wir den Eintrag 5:

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

sql = """SELECT * FROM telefonbuch"""

c.execute(sql)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen

for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

auswahl = int(input("Welchen Eintrag möchten Sie entfernen? Bitte die ID eingeben!"))

params = (auswahl,)
sql = """DELETE FROM telefonbuch WHERE id = ?"""
c.execute(sql, params)
conn.commit()
conn.close()

Führen wir jetzt die Codezelle aus, um den Inhalt der Datenbank anzuzeigen, stellen Sie fest, das der **Eintrag 3** *fehlt*.

## UPDATE
Frau Müller hat geheiratet und heißt jetzt Meier....

In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

name_neu = "Meier"
db_id = 4

params = (name_neu, db_id,)
sql = """UPDATE telefonbuch SET nachname = ? WHERE ID = ?"""

c.execute(sql, params)  # Der SQL-Befehl wird ausgeführt
ausgabe = c.fetchall()  # Suche ALLE Einträge, die dem Muster entsprechen
conn.commit()
conn.close()

for i, v, n, r, t in ausgabe:
    print(i, v, n, r, t)

Jetzt wieder die Zelle "ln 13" ausführen, und Änderung bewundern.

## DROP Table - Tabellen löschen

Als erstes erstellen wir uns schnell eine neue Tabelle, damit wir diese löschen können. ;-) <br><br>
Diese erstellen wir in derselben Datei:

In [ ]:
import sqlite3
conn = sqlite3.connect("telefonbuch.db")
c = conn.cursor()
sql = """CREATE TABLE IF NOT EXISTS Haustiere
(
    ID       integer
        constraint Liste_pk
            primary key,
    name  text,
    tierart text,
    farbe  text
)"""
c.execute(sql)
conn.commit()
conn.close()

Wie Ihnen vielleicht aufgefallen ist, nutzen wir in diesen Fall ein zusätzliches Statement "**TABLE IF NOT EXISTS**" Dieses Statement sorgt dafür, das die Tabelle nur dann erstellt wird, wenn diese NICHT bereits existiert.<br><br>Wenn Sie den Datenbankbrowser aktualisieren, sehen Sie die neue Datenbank: <br>
<img style="float: right;" src="img/sqlite9.png"><br>
Jetzt löschen wir diese wieder:


In [ ]:
import sqlite3

conn = sqlite3.connect("telefonbuch.db")  # Verbindung wird hergestellt
c = conn.cursor()  # Ein Cursor wird erstellt (Ermöglicht den Zugriff auf die Datenbank)

sql = """DROP TABLE IF EXISTS Haustiere"""
c.execute(sql)
conn.commit()
conn.close()

Und schon ist die Tabelle wieder weg.<br> <br>


#### Rollback
ROLLBACK ist ein zentraler Bestandteil des Transaktionssystems von SQLite, das die Datenintegrität sicherstellt (ACID-Eigenschaften).

In [1]:
import sqlite3

conn = sqlite3.connect('telefonbuch.db')
cursor = conn.cursor()

# Tabelle erstellen (falls nicht vorhanden)
cursor.execute('CREATE TABLE IF NOT EXISTS personen (name TEXT, age INTEGER)')
cursor.execute('DELETE FROM personen')  # Tabelle für das Beispiel leeren
conn.commit()  # Leeren bestätigen

try:
    # Eine Transaktion wird in Python automatisch gestartet, wenn man Daten ändert.

    print("Füge 'Alice' hinzu...")
    cursor.execute("INSERT INTO personen VALUES ('Alice', 30)")

    print("Füge 'Bob' hinzu...")
    cursor.execute("INSERT INTO personen VALUES ('Bob', 25)")

    # Stell dir vor, hier passiert ein Fehler oder eine Bedingung ist nicht erfüllt.
    # Wir entscheiden uns, die Änderungen zu verwerfen.
    cursor.execute('SELECT * FROM personen')
    print("\nInhalt der Tabelle vor dem Rollback:")
    for row in cursor.fetchall():
        print(row)  # Diese Schleife wird nichts ausgeben
    print("Ein Fehler ist aufgetreten! Führe Rollback aus...")
    conn.rollback()

except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")
    conn.rollback()

finally:
    # Überprüfen, ob die Daten wirklich nicht eingefügt wurden
    cursor.execute('SELECT * FROM personen')
    print("\nInhalt der Tabelle nach dem Rollback:")
    for row in cursor.fetchall():
        print(row)  # Diese Schleife wird nichts ausgeben

    conn.close()

Füge 'Alice' hinzu...
Füge 'Bob' hinzu...

Inhalt der Tabelle vor dem Rollback:
('Alice', 30)
('Bob', 25)
Ein Fehler ist aufgetreten! Führe Rollback aus...

Inhalt der Tabelle nach dem Rollback:


#### Kontextmanager
Die Vorteile sind:
1. Automatisches Schließen: Du musst conn.close() nicht mehr manuell aufrufen. Sobald der with-Block verlassen wird (auch durch einen Fehler), wird die Verbindung automatisch geschlossen.
2. Sauberer Code: Es ist sofort ersichtlich, in welchem Bereich die Datenbankverbindung aktiv ist.
Hinweis: Der Kontextmanager von sqlite3 kümmert sich primär um das Transaktionsmanagement (automatisches commit bei Erfolg, rollback bei Fehler), wenn man Daten ändert. Er schließt die Verbindung aber nicht immer automatisch in allen Python-Versionen so strikt, wie man es von Dateien kennt, aber es ist dennoch die empfohlene Vorgehensweise ("Best Practice"), da es den Code robuster macht. In neueren Python-Versionen verhält es sich wie erwartet.

In [ ]:
import sqlite3

# Kontextmanager für die Verbindung
with sqlite3.connect("telefonbuch.db") as conn:
    c = conn.cursor()

    sql = """SELECT * FROM telefonbuch"""
    c.execute(sql)

    ausgabe = c.fetchall()

# Die Verbindung wird hier automatisch geschlossen, auch bei Fehlern.

print(ausgabe)

# Iteration1:
for i in ausgabe:
    print(i)

## Fazit:
Wie Sie sehen sind Datenbanken unter Python keine große Schwierigkeit. Natürlich können wir an einem Vormittag nicht alle Befehle durchgehen, aber das ist eine gute Basis.<br>
In den Zusatzinformationen zu dieser Woche finden Sie einige Cheatsheets zum Thema Datenbank. Diese werden Ihnen sicherlich sehr behilflich sein.

## SQLite vs. MySQL etc.
Für diese Vorlesung und den dazugehörigen Demos reicht SQLite vollkommen aus. Auf Webservern werden Sie natürlich kein SQLite, sondern MySQL, MariaDB oder ähnliches finden. In der Nutzung unterscheidet sich da nicht sehr. 

Es stehen ein paar mehr Funktionen zur Verfügung.

**Eine Vorlesung zum Thema MySQL folgt...**

## Ein kleines Telefonbuch (Optional)

Wir schreiben ein kleines Telefonbuch:

Folgende Funktionen soll unser kleines Telefonbuch enthalten:
* hinzufügen eines Vornamen, Nachname und dazugehörige Rufnummer
* Nummer eines Eintrags ausgeben
* Einträge löschen
* das komplette Telefonbuch anzeigen lassen
* das Ganze über ein Textmenü
* objektorientiert

Das Menü könnte folgendermaßen aussehen:

Telefonbuch
Bitte wählen Sie ...
1 - Eintrag hinzufügen
2 - Nummer ausgeben
3 - Nummer löschen
4 - Liste anzeigen
e - Ende
Ihre Auswahl:


In [ ]:
import sqlite3
import sys
import os
from pathlib import Path
from druckst_du import wait_for_keypress


def datenbank_erstellen():
    with sqlite3.connect("telefonbuch.db") as conn:
        c = conn.cursor()
        sql = """create table if not exists telefonbuch
                 (
                     id        integer
                         constraint id
                             primary key,
                     vorname   TEXT,
                     nachname  TEXT,
                     vorwahl   TEXT,
                     rufnummer TEXT
                 ) \
              """
        c.execute(sql)
        conn.commit()

    menu()


def cls():
    os.system('cls' if os.name=='nt' else 'clear')


def eintrag_suchen():
    pass


def eintrag_hinzufuegen():
    vorname = input("Vorname: ")
    nachname = input("Nachname: ")
    vorwahl = input("Vorwahl: ")
    rufnummer = input("Rufnummer: ")

    # Überprüfen, ob alles ausgefüllt ist:
    if not vorname or nachname or vorwahl or rufnummer:
        print("Bitte alle Angaben ausfüllen!")
        wait_for_keypress()
        cls()
        menu()

    params = (vorname, nachname, vorwahl, rufnummer)

    try:
        with sqlite3.connect("telefonbuch.db") as conn:
            c = conn.cursor()
            sql = """INSERT INTO telefonbuch (vorname, nachname, vorwahl, rufnummer) VALUES (?, ?, ?, ?)"""
            c.execute(sql, params)
            conn.commit()

        print("Eintrag erfolgreich hinzugefügt!")

    except Exception as e:
        print("Es ist ein Fehler aufgetreten: \n", e)

    finally:
        wait_for_keypress()
        cls()
        menu()

def eintrag_loeschen():
    pass


def alles_ausgeben():
    pass


def beenden():
    print("Auf Wiedersehen!")
    sys.exit(0)


def menu():
    items = ["Eintrag suchen", "Eintrag hinzufügen", "Eintrag löschen", "Alle Einträge anzeigen", "Beenden"]
    for i, item in enumerate(items, 1):
        print(f"{i} - {item}")

    auswahl = input("Bitte wählen Sie aus: ")

    match auswahl:
        case "1":
            eintrag_suchen()

        case"2":
            eintrag_hinzufuegen()

        case "3":
            eintrag_loeschen()

        case "4":
            alles_ausgeben()

        case "5":
            beenden()

if __name__ == "__main__":
    # Pfad-Variable abhängig vom Skriptordner erstellen:
    ordner = Path(__file__).parent
    db_name = ordner / "telefonbuch.db"
    datenbank_erstellen()
    menu()

<img style="float: center;" src="img/wbs-logo.jpg">


### Abbildungs- und Quellenverzeichnis

Das Python Logo ist ein eingetragenes Warenzeichen der Python Software Foundation
Alle auf dieser Website veröffentlichten Logos sowie Marken-, Produkt- und Warenzeichen sind Eigentum der jeweiligen Unternehmen
© WBS TRAINING AG – Alle Rechte vorbehalten

### Nutzungsrechte:
Die Nutzung dieser Dokumentation ist ausschließlich für Schulungszwecke der WBS TRAINING AG gestattet. Eine Weitergabe an Dritte, auch auszugsweise, sowie Vervielfältigungen und Verbreitungen aller Art (elektronische und andere Verfahren) inklusive Übersetzungen sind nur mit vorheriger schriftlicher Zustimmung des Rechtinhabers gestattet. Zuwiderhandlungen verpflichten zu Schadenersatz.

### Herausgeber:

WBS TRAINING AG
Lorenzweg 5
12099 Berlin
Haftungsausschluss:
Alle Inhalte sind nach bestem Wissen korrekt und vollständig recherchiert und mit größtmöglicher Sorgfalt für die Schulungsunterlage zusammengestellt. Wir sind um die laufende Aktualisierung aller Informationen und Daten bemüht. Dennoch können Fehler (z.B. Abweichungen zur beschriebenen Hard- und Software durch kurzfristige Updates) auftreten, sodass wir für die vollständige Übereinstimmung, Richtigkeit und Aktualität keine Gewähr übernehmen. Hinweise unserer Nutzer werden konsequent weiterverfolgt.
